# LeetCode 331 — Verify Preorder Serialization of a Binary Tree bằng Python

Link đề: https://leetcode.com/problems/verify-preorder-serialization-of-a-binary-tree/ · Độ khó: **Medium** · Chủ đề: String, Stack, Tree, Binary Tree

Skeleton LeetCode:

```python
class Solution:
    def isValidSerialization(self, preorder: str) -> bool:
```

Ghi chú: đề, ví dụ và constraints lấy từ file đề có sẵn (bài này không phải Premium).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: một chuỗi tách bởi dấu phẩy, mỗi phần tử là số nguyên (nút) hoặc `#` (con trỏ rỗng); hỏi chuỗi có phải **duyệt tiền thứ tự hợp lệ** của một cây nhị phân hay không **mà không được dựng lại cây**.
2. Nhìn bài toán qua **khái niệm "chỗ trống" (slot)**: mỗi nút chiếm một chỗ trống, mỗi nút thật tạo thêm hai chỗ trống mới cho hai con, `#` không tạo thêm.
3. Đi từ **phân tích cú pháp đệ quy** (dựng ngầm cây, baseline) → **ngăn xếp rút gọn `x # #` thành `#`** → **đếm chỗ trống** `O(n)` thời gian, `O(1)` bộ nhớ phụ (cách nộp) → **ngăn xếp các bộ đếm con còn thiếu**.
4. Hiểu **vì sao phải kiểm tra "chỗ trống không âm" ở mọi bước** (không chỉ ở cuối) và **vì sao phải kết thúc đúng bằng `0`**.
5. Biết **sinh dữ liệu thử độc lập** bằng văn phạm `S → # | x S S` (liệt kê mọi chuỗi hợp lệ) để làm oracle.
6. Tránh các lỗi kinh điển: tách theo ký tự thay vì theo dấu phẩy, chỉ kiểm tra tổng ở cuối, rút gọn ngăn xếp không lặp lại, đếm lệch một.

## 2. Đề bài & ví dụ

Một cách tuần tự hóa cây nhị phân là dùng **duyệt tiền thứ tự**: gặp nút khác rỗng thì ghi giá trị; gặp nút rỗng thì ghi ký tự `#`. Ví dụ cây trong đề được tuần tự hóa thành `"9,3,4,#,#,1,#,#,2,#,6,#,#"`.

Cho một chuỗi các giá trị tách bởi dấu phẩy `preorder`, trả về `true` nếu đó là một chuỗi duyệt tiền thứ tự **hợp lệ** của một cây nhị phân.

Đảm bảo mỗi giá trị trong chuỗi là số nguyên hoặc `#`; định dạng đầu vào luôn hợp lệ (không có hai dấu phẩy liền nhau như `"1,,3"`). **Lưu ý:** không được dựng lại cây.

**Ví dụ 1:** `preorder = "9,3,4,#,#,1,#,#,2,#,6,#,#"` → `true`

**Ví dụ 2:** `preorder = "1,#"` → `false`

**Ví dụ 3:** `preorder = "9,#,#,1"` → `false`

**Ràng buộc:** `1 <= preorder.length <= 10^4`; `preorder` gồm các số nguyên trong `[0, 100]` và ký tự `#`, ngăn cách bởi dấu phẩy.

In [1]:
from typing import List
import random
import sys
import time

examples = [
    ("9,3,4,#,#,1,#,#,2,#,6,#,#", True),
    ("1,#", False),
    ("9,#,#,1", False),
]
for text, expected in examples:
    print(repr(text), "| các phần tử:", text.split(","), "| kỳ vọng:", expected)

'9,3,4,#,#,1,#,#,2,#,6,#,#' | các phần tử: ['9', '3', '4', '#', '#', '1', '#', '#', '2', '#', '6', '#', '#'] | kỳ vọng: True
'1,#' | các phần tử: ['1', '#'] | kỳ vọng: False
'9,#,#,1' | các phần tử: ['9', '#', '#', '1'] | kỳ vọng: False


## 3. Kiến thức nền

### 3.1. Duyệt tiền thứ tự (preorder)

Tiền thứ tự: ghi **nút gốc**, rồi duyệt **cây con trái**, rồi **cây con phải**. Với con trỏ rỗng ta ghi `#`. Vì mỗi nút thật có đúng hai con (có thể rỗng), một chuỗi hợp lệ có dạng đệ quy:

`S → #` hoặc `S → x S S`.

Tức là một cây hoặc rỗng, hoặc là một nút gốc rồi hai cây con.
#### Code demo

In [2]:
def serialize(tree):
    """tree = None hoặc (val, left, right); trả về chuỗi tiền thứ tự có '#'."""
    if tree is None:
        return "#"
    val, left, right = tree
    return f"{val},{serialize(left)},{serialize(right)}"


tree = (9, (3, (4, None, None), (1, None, None)), (2, None, (6, None, None)))
print(serialize(tree))                       # đúng chuỗi ví dụ 1
print(serialize(None), "|", serialize((5, None, None)))

9,3,4,#,#,1,#,#,2,#,6,#,#
# | 5,#,#


### 3.2. Đếm nút thật và nút rỗng: `rỗng = thật + 1`

Với mọi cây nhị phân có `a` nút thật, tổng số con trỏ (hai trên mỗi nút) là `2a`; trong đó `a - 1` con trỏ nối tới các nút không phải gốc, còn lại `2a - (a - 1) = a + 1` con trỏ là rỗng. Vậy chuỗi hợp lệ có **số `#` = số nút thật + 1**. Đây chỉ là điều kiện **cần**: chuỗi `"#,#,1"` thỏa nhưng không hợp lệ (cây đã kết thúc ngay sau `#` đầu tiên). Nó cho trực giác "đếm", nhưng cần thêm điều kiện **theo từng bước**.
#### Code demo

In [3]:
for text, expected in examples:
    items = text.split(",")
    nulls = items.count("#")
    nodes = len(items) - nulls
    print(f"{text!r:<32} nút thật={nodes} rỗng={nulls} | rỗng == thật+1: {nulls == nodes + 1} | hợp lệ: {expected}")
print("nhưng '#,#,1' cũng có rỗng == thật+1:", "#,#,1".split(",").count("#") == 1 + 1)

'9,3,4,#,#,1,#,#,2,#,6,#,#'      nút thật=6 rỗng=7 | rỗng == thật+1: True | hợp lệ: True
'1,#'                            nút thật=1 rỗng=1 | rỗng == thật+1: False | hợp lệ: False
'9,#,#,1'                        nút thật=2 rỗng=2 | rỗng == thật+1: False | hợp lệ: False
nhưng '#,#,1' cũng có rỗng == thật+1: True


### 3.3. Khái niệm "chỗ trống" (slot)

Hình dung chuỗi như đang **lấp đầy các chỗ trống**. Ban đầu có **1 chỗ trống** (vị trí của gốc). Đọc mỗi phần tử:

- Phần tử đó **lấp một chỗ trống**: `slots -= 1` (nếu không còn chỗ trống thì chuỗi sai).
- Nếu là **nút thật** thì nó tạo **hai chỗ trống mới** cho hai con: `slots += 2`.
- Nếu là `#` thì không tạo chỗ trống mới.

Chuỗi hợp lệ ⟺ không bao giờ hết chỗ trống **trước** khi đọc xong và kết thúc với **đúng `0`** chỗ trống.
#### Code demo

In [4]:
def trace_slots_quick(text):
    slots = 1
    log = []
    for tok in text.split(","):
        slots -= 1
        if slots < 0:
            log.append((tok, "hết chỗ"))
            break
        if tok != "#":
            slots += 2
        log.append((tok, slots))
    return log


print(trace_slots_quick("1,#,#"))

[('1', 2), ('#', 1), ('#', 0)]


### 3.4. Chuỗi tuần tự hóa và `split(",")`

Phải tách theo **dấu phẩy**, vì giá trị có thể có nhiều chữ số (`10`, `100`). Tách theo từng ký tự sẽ chặt `100` thành `1`, `0`, `0` — ba "nút" thay vì một.
#### Code demo

In [5]:
text = "10,#,#"
print(text.split(","), "| tách sai theo ký tự:", [c for c in text if c != ","])

['10', '#', '#'] | tách sai theo ký tự: ['1', '0', '#', '#']


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

`"9,3,4,#,#,1,#,#,2,#,6,#,#"` với số chỗ trống sau mỗi phần tử (bắt đầu 1):

| đọc | chỗ trống trước | `-1` | nút thật `+2` | chỗ trống sau |
|---|---|---|---|---|
| `9` | 1 | 0 | +2 | 2 |
| `3` | 2 | 1 | +2 | 3 |
| `4` | 3 | 2 | +2 | 4 |
| `#` | 4 | 3 | — | 3 |
| `#` | 3 | 2 | — | 2 |
| `1` | 2 | 1 | +2 | 3 |
| `#` | 3 | 2 | — | 2 |
| `#` | 2 | 1 | — | 1 |
| `2` | 1 | 0 | +2 | 2 |
| `#` | 2 | 1 | — | 1 |
| `6` | 1 | 0 | +2 | 2 |
| `#` | 2 | 1 | — | 1 |
| `#` | 1 | 0 | — | **0** |

Kết thúc đúng `0` và không bước nào âm ⇒ **hợp lệ**.

- `"1,#"`: `1 → 2`, `# → 1` ⇒ còn 1 chỗ trống ⇒ **thiếu** nhánh ⇒ sai.
- `"9,#,#,1"`: `9 → 2`, `# → 1`, `# → 0`, rồi đọc `1` khi chỗ trống = 0 ⇒ **thừa** ⇒ sai.

## 5. Cách 1 — Brute Force / baseline: phân tích cú pháp đệ quy (dựng ngầm cây)

**Ý tưởng.** Bám sát văn phạm `S → # | x S S`: hàm `parse(i)` đọc phần tử ở vị trí `i`; nếu là `#` thì trả về `i + 1`; nếu là nút thật thì đọc cây con trái rồi cây con phải. Chuỗi hợp lệ ⟺ `parse(0)` hợp lệ và dùng hết **đúng** mọi phần tử.

(Cách này **tương đương việc dựng cây** — đề cấm; ta chỉ dùng như baseline và để đối chiếu.)
#### Code demo

In [6]:
sys.setrecursionlimit(50_000)


def valid_parse(preorder):
    items = preorder.split(",")
    n = len(items)

    def parse(i):
        """Đọc một cây bắt đầu tại i; trả về vị trí ngay sau cây, hoặc -1 nếu lỗi."""
        if i >= n:
            return -1
        if items[i] == "#":
            return i + 1
        j = parse(i + 1)                  # cây con trái
        if j == -1:
            return -1
        return parse(j)                   # cây con phải

    return parse(0) == n


for text, expected in examples:
    print(repr(text), "->", valid_parse(text), "| kỳ vọng:", expected)

'9,3,4,#,#,1,#,#,2,#,6,#,#' -> True | kỳ vọng: True
'1,#' -> False | kỳ vọng: False
'9,#,#,1' -> False | kỳ vọng: False


**Độ phức tạp (Cách 1).** **Time `O(n)`** (mỗi phần tử được đọc một lần), **Space `O(h)`** ngăn xếp đệ quy (`h` là độ sâu, tối đa `~ n/2` với cây "xương cá").

**Nút thắt.** Dùng đệ quy sâu và về bản chất là dựng cây; ta muốn một vòng lặp đơn giản, `O(1)` phụ.

## 6. Ý tưởng thuật toán / bước đột phá

Dùng **đếm chỗ trống**:

```
slots = 1
với mỗi phần tử:
    slots -= 1
    nếu slots < 0: return False       # hết chỗ trước khi đọc xong
    nếu là nút thật: slots += 2
return slots == 0
```

Mỗi phần tử tiêu thụ đúng một "đầu vào" và nút thật sinh hai "đầu ra"; đây chính là **cân bằng bậc vào/bậc ra** của cây. Chỉ cần một biến đếm, `O(n)` thời gian, `O(1)` phụ.

## 7. Các cách tiếp theo

### Cách 2 — Ngăn xếp rút gọn: thay `x # #` bằng `#`

**Giải thích.** Duyệt từng phần tử, đẩy vào ngăn xếp. Mỗi khi đỉnh ngăn xếp là `[..., x, #, #]` với `x` là **nút thật**, thì **cây con** đó đã hoàn chỉnh: thay ba phần tử đó bằng một `#` (coi cả cây con như một con trỏ rỗng đã lấp). **Lặp lại** việc rút gọn (vì rút gọn có thể kích hoạt rút gọn tiếp). Cuối cùng chuỗi hợp lệ ⟺ ngăn xếp đúng bằng `["#"]`.
#### Code demo

In [7]:
def valid_stack_reduce(preorder):
    stack = []
    for tok in preorder.split(","):
        stack.append(tok)
        while len(stack) >= 3 and stack[-1] == "#" and stack[-2] == "#" and stack[-3] != "#":
            stack.pop()
            stack.pop()
            stack.pop()
            stack.append("#")
    return stack == ["#"]


for text, expected in examples:
    print(repr(text), "->", valid_stack_reduce(text), "| kỳ vọng:", expected)

'9,3,4,#,#,1,#,#,2,#,6,#,#' -> True | kỳ vọng: True
'1,#' -> False | kỳ vọng: False
'9,#,#,1' -> False | kỳ vọng: False


**Độ phức tạp Cách 2:** **Time `O(n)`** (mỗi phần tử đẩy một lần, rút gọn tối đa `n/2` lần, mỗi lần pop 3 push 1: tổng phép là `O(n)`), **Space `O(n)`** (ngăn xếp).

### Cách 3 — Đếm chỗ trống (cách nộp chuẩn)

**Giải thích.** Như mục 6.
#### Code demo

In [8]:
def valid_slots(preorder):
    slots = 1
    for tok in preorder.split(","):
        slots -= 1                        # phần tử này lấp một chỗ trống
        if slots < 0:                     # không còn chỗ => thừa phần tử
            return False
        if tok != "#":
            slots += 2                    # nút thật mở thêm hai chỗ trống
    return slots == 0                     # phải lấp đầy đúng


for text, expected in examples:
    print(repr(text), "->", valid_slots(text), "| kỳ vọng:", expected)

'9,3,4,#,#,1,#,#,2,#,6,#,#' -> True | kỳ vọng: True
'1,#' -> False | kỳ vọng: False
'9,#,#,1' -> False | kỳ vọng: False


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(1)`** phụ (chưa tính `split`; có thể duyệt chuỗi bằng chỉ số để không tạo danh sách).

### Cách 4 — Ngăn xếp các bộ đếm "số con còn thiếu"

**Giải thích.** Mô phỏng duyệt cây bằng ngăn xếp: mỗi nút thật đang "mở" có một bộ đếm **số con chưa nhận**. Khi đọc một phần tử: nếu ngăn xếp rỗng mà chưa phải phần tử đầu ⇒ cây đã hoàn chỉnh từ trước ⇒ **thừa** ⇒ sai. Ngược lại phần tử đó là con của nút ở đỉnh: giảm bộ đếm đỉnh; nếu nút mới là nút thật thì đẩy bộ đếm `2`; sau đó **pop mọi nút đã đủ hai con** (bộ đếm `0`) — việc pop lan truyền lên các tổ tiên.
#### Code demo

In [9]:
def valid_counter_stack(preorder):
    stack = []                            # mỗi phần tử: số con còn thiếu của một nút thật đang mở
    for i, tok in enumerate(preorder.split(",")):
        if i > 0 and not stack:           # cây đã đủ mà vẫn còn phần tử
            return False
        if stack:
            stack[-1] -= 1                # phần tử này là một con của nút ở đỉnh
        if tok != "#":
            stack.append(2)               # nút thật mới cần hai con
        while stack and stack[-1] == 0:   # nút đủ hai con thì đóng, lan truyền lên trên
            stack.pop()
    return not stack                      # hợp lệ nếu mọi nút mở đã đóng


for text, expected in examples:
    print(repr(text), "->", valid_counter_stack(text), "| kỳ vọng:", expected)

'9,3,4,#,#,1,#,#,2,#,6,#,#' -> True | kỳ vọng: True
'1,#' -> False | kỳ vọng: False
'9,#,#,1' -> False | kỳ vọng: False


**Độ phức tạp Cách 4:** **Time `O(n)`** (mỗi phần tử đẩy/pop tối đa một lần), **Space `O(h)`** (độ sâu cây đang mở).

> Cách 3 là bản nộp chuẩn: ngắn nhất và `O(1)` phụ. Cách 2 và 4 giúp hiểu cấu trúc cây; Cách 1 là cách đệ quy trực quan nhưng bị cấm vì dựng cây.

## 8. TRACE từng bước

Ta trace Cách 3 (đếm chỗ trống) trên ví dụ 1, ví dụ 3, và `"#,#,1"` (điều kiện đếm tổng đúng nhưng chuỗi sai), in `slots` sau mỗi phần tử; và Cách 2 (rút gọn) trên ví dụ 1.
#### Code demo

In [10]:
def trace_slots(preorder):
    slots = 1
    print(f"preorder = {preorder!r} | slots ban đầu = 1")
    for k, tok in enumerate(preorder.split(",")):
        before = slots
        slots -= 1
        if slots < 0:
            print(f"  #{k:<2} đọc {tok:>2}: {before} - 1 = {slots} < 0 => THỪA, trả False")
            return False
        note = ""
        if tok != "#":
            slots += 2
            note = " (+2 vì là nút thật)"
        print(f"  #{k:<2} đọc {tok:>2}: {before} - 1{note} => slots = {slots}")
    print(f"  kết thúc: slots = {slots} => {'hợp lệ' if slots == 0 else 'THIẾU nhánh'}")
    return slots == 0


trace_slots("9,3,4,#,#,1,#,#,2,#,6,#,#")
print()
trace_slots("9,#,#,1")
print()
trace_slots("#,#,1")

preorder = '9,3,4,#,#,1,#,#,2,#,6,#,#' | slots ban đầu = 1
  #0  đọc  9: 1 - 1 (+2 vì là nút thật) => slots = 2
  #1  đọc  3: 2 - 1 (+2 vì là nút thật) => slots = 3
  #2  đọc  4: 3 - 1 (+2 vì là nút thật) => slots = 4
  #3  đọc  #: 4 - 1 => slots = 3
  #4  đọc  #: 3 - 1 => slots = 2
  #5  đọc  1: 2 - 1 (+2 vì là nút thật) => slots = 3
  #6  đọc  #: 3 - 1 => slots = 2
  #7  đọc  #: 2 - 1 => slots = 1
  #8  đọc  2: 1 - 1 (+2 vì là nút thật) => slots = 2
  #9  đọc  #: 2 - 1 => slots = 1
  #10 đọc  6: 1 - 1 (+2 vì là nút thật) => slots = 2
  #11 đọc  #: 2 - 1 => slots = 1
  #12 đọc  #: 1 - 1 => slots = 0
  kết thúc: slots = 0 => hợp lệ

preorder = '9,#,#,1' | slots ban đầu = 1
  #0  đọc  9: 1 - 1 (+2 vì là nút thật) => slots = 2
  #1  đọc  #: 2 - 1 => slots = 1
  #2  đọc  #: 1 - 1 => slots = 0
  #3  đọc  1: 0 - 1 = -1 < 0 => THỪA, trả False

preorder = '#,#,1' | slots ban đầu = 1
  #0  đọc  #: 1 - 1 => slots = 0
  #1  đọc  #: 0 - 1 = -1 < 0 => THỪA, trả False


False

In [11]:
def trace_reduce(preorder):
    stack = []
    print(f"preorder = {preorder!r}")
    for tok in preorder.split(","):
        stack.append(tok)
        msg = f"đẩy {tok:>2} -> {stack}"
        while len(stack) >= 3 and stack[-1] == "#" and stack[-2] == "#" and stack[-3] != "#":
            del stack[-3:]
            stack.append("#")
            msg += f"  | rút gọn x##->#: {stack}"
        print("  " + msg)
    print("  kết quả:", stack, "=> hợp lệ" if stack == ["#"] else "=> không hợp lệ")


trace_reduce("9,3,4,#,#,1,#,#,2,#,6,#,#")

preorder = '9,3,4,#,#,1,#,#,2,#,6,#,#'
  đẩy  9 -> ['9']
  đẩy  3 -> ['9', '3']
  đẩy  4 -> ['9', '3', '4']
  đẩy  # -> ['9', '3', '4', '#']
  đẩy  # -> ['9', '3', '4', '#', '#']  | rút gọn x##->#: ['9', '3', '#']
  đẩy  1 -> ['9', '3', '#', '1']
  đẩy  # -> ['9', '3', '#', '1', '#']
  đẩy  # -> ['9', '3', '#', '1', '#', '#']  | rút gọn x##->#: ['9', '3', '#', '#']  | rút gọn x##->#: ['9', '#']
  đẩy  2 -> ['9', '#', '2']
  đẩy  # -> ['9', '#', '2', '#']
  đẩy  6 -> ['9', '#', '2', '#', '6']
  đẩy  # -> ['9', '#', '2', '#', '6', '#']
  đẩy  # -> ['9', '#', '2', '#', '6', '#', '#']  | rút gọn x##->#: ['9', '#', '2', '#', '#']  | rút gọn x##->#: ['9', '#', '#']  | rút gọn x##->#: ['#']
  kết quả: ['#'] => hợp lệ


Bảng tóm tắt (ví dụ 3 `"9,#,#,1"`):

| đọc | `slots` trước | sau `-1` | sau khi cộng | ghi chú |
|---|---|---|---|---|
| `9` | 1 | 0 | 2 | nút thật |
| `#` | 2 | 1 | 1 | rỗng |
| `#` | 1 | 0 | 0 | rỗng, cây đã đủ |
| `1` | 0 | **-1** | — | **thừa**, trả `False` |

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3. Đúng `class Solution`, đúng `isValidSerialization(self, preorder: str) -> bool`. Để không tạo danh sách phụ, ta duyệt chuỗi bằng chỉ số và nhận biết từng phần tử theo dấu phẩy.
#### Code demo

In [12]:
class Solution:
    def isValidSerialization(self, preorder: str) -> bool:
        slots = 1                                   # số chỗ trống đang chờ được lấp
        for tok in preorder.split(","):
            slots -= 1                              # phần tử hiện tại lấp một chỗ
            if slots < 0:                           # hết chỗ trước khi đọc xong
                return False
            if tok != "#":
                slots += 2                          # nút thật mở hai chỗ cho hai con
        return slots == 0                           # đúng bằng 0: cây vừa khít


sol = Solution()
for text, expected in examples:
    assert sol.isValidSerialization(text) == expected, (text, expected)
print("OK: 3 ví dụ của đề")

OK: 3 ví dụ của đề


## 10. Vì sao thuật toán đúng

- **Văn phạm:** chuỗi hợp lệ là chuỗi sinh bởi `S → # | x S S`. Định nghĩa `need(i)` là số chỗ trống đang chờ sau khi đọc `i` phần tử đầu; ban đầu `need = 1` (một `S` chờ được sinh).
- **Mỗi phần tử thay một `S` đang chờ:** đọc `#` dùng luật `S → #` (số `S` chờ giảm 1); đọc `x` dùng luật `S → x S S` (giảm 1 rồi tăng 2). Đây đúng là phép `slots -= 1`, rồi `slots += 2` nếu là nút thật.
- **Tiền thứ tự cho phép đếm không cần biết "S nào":** trong tiền thứ tự, việc `S` nào được thay tiếp theo là xác định (cái ngoài cùng bên trái chưa thay) — nên chỉ cần **số lượng** `S` đang chờ, không cần vị trí.
- **Điều kiện đúng/sai:** chuỗi được sinh hoàn toàn ⟺ sau khi đọc hết, không còn `S` chờ (`slots == 0`) và **không có bước nào** đọc khi `slots == 0` (vì lúc đó không còn `S` để thay: `slots` sẽ xuống `-1`). Nếu chỉ kiểm tra cuối cùng mà quên kiểm tra giữa chừng, `"#,#,1"` bị nhận nhầm (xem mục 13).
- **Cách 2:** `x # #` là một cây con hoàn chỉnh; thay bằng `#` không đổi tính hợp lệ của chuỗi (một cây con hoàn chỉnh chiếm đúng một chỗ trống, như `#`); lặp lại đến khi không rút gọn được nữa, chuỗi hợp lệ thu về `#`.
- **Cách 4:** mô phỏng đúng duyệt cây: bộ đếm của mỗi nút thật là số con còn thiếu; nút đóng khi đủ hai con.

## 11. Phân tích Time / Space Complexity

Gọi `n` là số phần tử (≤ `5·10^3` vì chuỗi dài tối đa `10^4` ký tự gồm cả dấu phẩy).

| Cách | Time | Space |
|---|---|---|
| 1. Phân tích cú pháp đệ quy | `O(n)` | `O(h)` ngăn xếp (tối đa `~ n/2`) |
| 2. Ngăn xếp rút gọn | `O(n)` | `O(n)` |
| 3. Đếm chỗ trống | `O(n)` | `O(1)` phụ (+ `split`) |
| 4. Ngăn xếp bộ đếm | `O(n)` | `O(h)` |

Với `n ≈ 5·10^3`, mọi cách chạy trong tích tắc. Điểm khác nhau chủ yếu là bộ nhớ và độ sâu đệ quy: Cách 1 với chuỗi "xương cá" sâu `~ 2500` đã vượt giới hạn đệ quy mặc định 1000 của Python.

## 12. Bộ test

`oracle(text)` **độc lập**: liệt kê mọi chuỗi hợp lệ (dạng hình `n`/`#`) theo văn phạm `S → # | x S S` cho các độ dài lẻ nhỏ rồi **kiểm tra có thuộc tập đó không**. Cách này không dùng đếm hay ngăn xếp. Dùng cho chuỗi tối đa `13` phần tử; chuỗi lớn kiểm tra bằng **tính chất** (dựng cây ngẫu nhiên rồi tuần tự hóa phải trả `True`, v.v.).
#### Code demo

In [13]:
def build_valid_shapes(max_len):
    """valid[L] = tập các chuỗi hình ('n' = nút thật, '#' = rỗng) hợp lệ có đúng L phần tử."""
    valid = {1: {("#",)}}
    for L in range(3, max_len + 1, 2):
        cur = set()
        for la in range(1, L - 1, 2):                 # độ dài cây con trái (lẻ)
            lb = L - 1 - la
            for a in valid[la]:
                for b in valid[lb]:
                    cur.add(("n",) + a + b)
        valid[L] = cur
    return valid


VALID = build_valid_shapes(13)
print("số chuỗi hợp lệ theo độ dài:", {L: len(v) for L, v in sorted(VALID.items())})   # số Catalan 1,1,2,5,14,42,132


def oracle(text):
    items = text.split(",")
    shape = tuple("#" if t == "#" else "n" for t in items)
    return shape in VALID.get(len(shape), set())


impls = {
    "parse": valid_parse,
    "stack_reduce": valid_stack_reduce,
    "slots": valid_slots,
    "counter_stack": valid_counter_stack,
    "submit": sol.isValidSerialization,
}

for text, expected in examples:
    assert oracle(text) == expected
    for key, fn in impls.items():
        assert fn(text) == expected, (key, text, expected, fn(text))
print("PASS ví dụ của đề (cả oracle)")

số chuỗi hợp lệ theo độ dài: {1: 1, 3: 1, 5: 2, 7: 5, 9: 14, 11: 42, 13: 132}
PASS ví dụ của đề (cả oracle)


### 12.1. Các trường hợp biên
#### Code demo

In [14]:
edge_cases = [
    ("#", True, "chỉ một con trỏ rỗng (cây rỗng)"),
    ("1,#,#", True, "một nút, hai con rỗng"),
    ("1,#", False, "thiếu một con"),
    ("1", False, "chỉ một nút thật (thiếu cả hai con)"),
    ("#,#", False, "thừa sau cây rỗng"),
    ("#,#,1", False, "rỗng = thật + 1 nhưng cây đã kết thúc"),
    ("1,#,#,#", False, "thừa một rỗng"),
    ("1,#,#,1,#,#", False, "hai cây liền nhau (rừng, không phải một cây)"),
    ("1,2,#,#,#", True, "nút 2 là con trái, con phải rỗng"),
    ("1,#,2,#,#", True, "nút 2 là con phải"),
    ("1,2,3,#,#,#,#", True, "cây xương cá trái"),
    ("1,2,3,4,#,#,#,#,#", True, "cây xương cá trái độ sâu 4"),
    ("100,0,#,#,#", True, "giá trị nhiều chữ số và 0"),
    ("10,#,#", True, "giá trị hai chữ số (phải tách theo dấu phẩy)"),
    ("9,3,4,#,#,1,#,#,2,#,6,#,#", True, "ví dụ 1"),
]
for text, expected, label in edge_cases:
    assert oracle(text) == expected, (label, "oracle sai kỳ vọng tay", text)
    for key, fn in impls.items():
        assert fn(text) == expected, (key, label, text, expected, fn(text))
    print("PASS", label)

PASS chỉ một con trỏ rỗng (cây rỗng)
PASS một nút, hai con rỗng
PASS thiếu một con
PASS chỉ một nút thật (thiếu cả hai con)
PASS thừa sau cây rỗng
PASS rỗng = thật + 1 nhưng cây đã kết thúc
PASS thừa một rỗng
PASS hai cây liền nhau (rừng, không phải một cây)
PASS nút 2 là con trái, con phải rỗng
PASS nút 2 là con phải
PASS cây xương cá trái
PASS cây xương cá trái độ sâu 4
PASS giá trị nhiều chữ số và 0
PASS giá trị hai chữ số (phải tách theo dấu phẩy)
PASS ví dụ 1


### 12.2. Tính chất và phản ví dụ

- Cây dựng ngẫu nhiên rồi tuần tự hóa ⇒ **luôn `True`**.
- Với chuỗi hợp lệ: **bỏ phần tử cuối** ⇒ `False` (thiếu); **thêm `#` cuối** ⇒ `False` (thừa).
- Số `#` trong chuỗi hợp lệ = số nút thật + 1.
- Mọi **tiền tố thực sự** của một chuỗi hợp lệ đều **không** hợp lệ (một chuỗi hợp lệ không thể là tiền tố của chuỗi hợp lệ khác, vì cây đã kết thúc).
- **Phản ví dụ** cho điều kiện "chỉ đếm tổng": `"#,#,1"`.
#### Code demo

In [15]:
def random_tree(rng, max_nodes):
    """Dựng cây ngẫu nhiên (val, left, right) hoặc None với tối đa max_nodes nút thật."""
    if max_nodes == 0 or rng.random() < 0.25:
        return None
    left_nodes = rng.randint(0, max_nodes - 1)
    return (rng.randint(0, 100), random_tree(rng, left_nodes), random_tree(rng, max_nodes - 1 - left_nodes))


rng = random.Random(331)
for _ in range(500):
    text = serialize(random_tree(rng, rng.randint(0, 8)))
    items = text.split(",")
    for key, fn in impls.items():
        assert fn(text) is True, (key, text)
    assert items.count("#") == len(items) - items.count("#") + 1
    if len(items) > 1:
        assert not sol.isValidSerialization(",".join(items[:-1]))          # thiếu
        for cut in range(1, len(items)):
            assert not sol.isValidSerialization(",".join(items[:cut])), (text, cut)   # mọi tiền tố thật sự đều sai
    assert not sol.isValidSerialization(text + ",#")                          # thừa
print("PASS: dựng cây -> tuần tự hóa luôn True; bớt phần tử cuối / thêm # / mọi tiền tố thật sự đều False")


def sai_chi_dem_tong(preorder):
    items = preorder.split(",")
    nulls = items.count("#")
    return nulls == len(items) - nulls + 1                 # SAI: chỉ so sánh tổng


print("'#,#,1' chỉ đếm tổng:", sai_chi_dem_tong("#,#,1"), "| đúng:", sol.isValidSerialization("#,#,1"))
assert sai_chi_dem_tong("#,#,1") != sol.isValidSerialization("#,#,1")

PASS: dựng cây -> tuần tự hóa luôn True; bớt phần tử cuối / thêm # / mọi tiền tố thật sự đều False
'#,#,1' chỉ đếm tổng: True | đúng: False


### 12.3. Kiểm tra ngẫu nhiên (có seed) và dữ liệu lớn
#### Code demo

In [16]:
rng = random.Random(2331)
checked = [0, 0]
for trial in range(6000):
    L = rng.choice([1, 3, 5, 7, 9, 11, 13]) if rng.random() < 0.7 else rng.randint(1, 13)
    p_null = rng.choice([0.4, 0.5, 0.6])
    items = ["#" if rng.random() < p_null else str(rng.randint(0, 100)) for _ in range(L)]
    text = ",".join(items)
    expected = oracle(text)
    checked[expected] += 1
    for key, fn in impls.items():
        assert fn(text) == expected, (key, text, expected, fn(text))
print(f"PASS 6000 chuỗi ngẫu nhiên so với oracle văn phạm (hợp lệ: {checked[1]}, không hợp lệ: {checked[0]})")
assert checked[1] > 100 and checked[0] > 100

# Dữ liệu lớn: chuỗi dài tối đa 10^4 ký tự
big_chain = ",".join(["1"] * 2499 + ["#"] * 2500)           # xương cá trái, sâu 2499
big_right = ",".join(["1", "#"] * 2499 + ["#"])             # xương cá phải


def gen_valid_big(rng, limit):
    """Sinh chuỗi hợp lệ dài ~limit phần tử bằng chính quy tắc chỗ trống (chỉ để tạo dữ liệu)."""
    toks, slots = [], 1
    while slots:
        if len(toks) + slots >= limit or (len(toks) >= 30 and rng.random() < 0.45):
            toks.append("#")
            slots -= 1
        else:
            toks.append(str(rng.randint(0, 100)))
            slots += 1
    return ",".join(toks)


big_random = gen_valid_big(random.Random(77), 2400)
big_cases = [
    ("xương cá trái", big_chain, True),
    ("xương cá phải", big_right, True),
    ("cây ngẫu nhiên lớn", big_random, True),
    ("xương cá trái thiếu 1 phần tử", big_chain.rsplit(",", 1)[0], False),
    ("xương cá trái thừa 1 '#'", big_chain + ",#", False),
]
for label, text, expected in big_cases:
    assert len(text) <= 10_000 + 10, len(text)
    row = []
    for key, fn in impls.items():
        t0 = time.perf_counter()
        r = fn(text)
        dt = (time.perf_counter() - t0) * 1000
        assert r == expected, (label, key, r, expected)
        row.append(f"{key}={dt:6.2f}ms")
    print(f"{label:<32} len={len(text):<6} | " + " ".join(row))
print("PASS dữ liệu lớn")

PASS 6000 chuỗi ngẫu nhiên so với oracle văn phạm (hợp lệ: 595, không hợp lệ: 5405)
xương cá trái                    len=9997   | parse=  0.68ms stack_reduce=  0.60ms slots=  0.26ms counter_stack=  0.61ms submit=  0.23ms
xương cá phải                    len=9997   | parse=  0.54ms stack_reduce=  0.84ms slots=  0.22ms counter_stack=  0.61ms submit=  0.22ms
cây ngẫu nhiên lớn               len=5880   | parse=  0.25ms stack_reduce=  0.32ms slots=  0.16ms counter_stack=  0.32ms submit=  0.14ms
xương cá trái thiếu 1 phần tử    len=9995   | parse=  0.55ms stack_reduce=  0.57ms slots=  0.24ms counter_stack=  0.58ms submit=  0.23ms
xương cá trái thừa 1 '#'         len=9999   | parse=  0.52ms stack_reduce=  0.55ms slots=  0.24ms counter_stack=  0.58ms submit=  0.23ms
PASS dữ liệu lớn


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Tách theo ký tự thay vì theo dấu phẩy
#### Code sai

In [17]:
def sai_tach_ky_tu(preorder):
    items = [c for c in preorder if c != ","]               # SAI: "10" thành '1', '0'
    slots = 1
    for tok in items:
        slots -= 1
        if slots < 0:
            return False
        if tok != "#":
            slots += 2
    return slots == 0


text = "10,#,#"
print("sai:", sai_tach_ky_tu(text), "| đúng:", sol.isValidSerialization(text))
assert sai_tach_ky_tu(text) != sol.isValidSerialization(text)

sai: False | đúng: True


**Vì sao sai:** `10` là **một** nút nhưng bị chặt thành hai nút `1` và `0`, làm số chỗ trống lệch. **Code đúng:** `preorder.split(",")`.

### Lỗi 2: Chỉ kiểm tra `slots == 0` ở cuối, không kiểm tra âm giữa chừng
#### Code sai

In [18]:
def sai_khong_kiem_tra_am(preorder):
    slots = 1
    for tok in preorder.split(","):
        slots -= 1                                          # SAI: thiếu 'if slots < 0: return False'
        if tok != "#":
            slots += 2
    return slots == 0


text = "#,#,1"
print("sai:", sai_khong_kiem_tra_am(text), "| đúng:", sol.isValidSerialization(text))
assert sai_khong_kiem_tra_am(text) != sol.isValidSerialization(text)

sai: True | đúng: False


**Vì sao sai:** ở `"#,#,1"` chỗ trống đi `1 → 0 → -1 → 0`; cuối cùng bằng `0` nhưng giữa chừng đã âm (đọc phần tử khi không còn chỗ). **Code đúng:** `if slots < 0: return False` ngay sau khi trừ.

### Lỗi 3: Rút gọn ngăn xếp chỉ một lần (không lặp)
#### Code sai

In [19]:
def sai_rut_gon_mot_lan(preorder):
    stack = []
    for tok in preorder.split(","):
        stack.append(tok)
        if len(stack) >= 3 and stack[-1] == "#" and stack[-2] == "#" and stack[-3] != "#":   # SAI: 'if' thay vì 'while'
            del stack[-3:]
            stack.append("#")
    return stack == ["#"]


text = "9,3,4,#,#,1,#,#,2,#,6,#,#"
print("sai:", sai_rut_gon_mot_lan(text), "| đúng:", sol.isValidSerialization(text))
assert sai_rut_gon_mot_lan(text) != sol.isValidSerialization(text)

sai: False | đúng: True


**Vì sao sai:** sau một lần rút gọn, đỉnh ngăn xếp có thể lại thành `x # #` (cây cha vừa đủ hai con) và cần rút gọn tiếp; một `if` bỏ sót. **Code đúng:** dùng `while` để rút gọn lan truyền.

### Lỗi 4: Khởi tạo chỗ trống bằng 0 (lệch một)
#### Code sai

In [20]:
def sai_slots_bang_0(preorder):
    slots = 0                                              # SAI: phải là 1 (một chỗ cho nút gốc)
    for tok in preorder.split(","):
        slots -= 1
        if slots < 0:
            return False
        if tok != "#":
            slots += 2
    return slots == 0


for text in ("#", "1,#,#"):
    print(repr(text), "sai:", sai_slots_bang_0(text), "| đúng:", sol.isValidSerialization(text))
    assert sai_slots_bang_0(text) != sol.isValidSerialization(text)

'#' sai: False | đúng: True
'1,#,#' sai: False | đúng: True


**Vì sao sai:** trước khi đọc gì, cây cần **một** chỗ cho nút gốc. Khởi tạo `0` làm mọi chuỗi bị loại ở phần tử đầu. **Code đúng:** `slots = 1`.

### Lỗi 5: Dùng đệ quy mà không nâng giới hạn đệ quy
#### Code sai

In [21]:
def goi_parse_voi_gioi_han(text, limit):
    old = sys.getrecursionlimit()
    sys.setrecursionlimit(limit)
    try:
        return valid_parse(text)
    except RecursionError:
        return "RecursionError"
    finally:
        sys.setrecursionlimit(old)


chain = ",".join(["1"] * 2499 + ["#"] * 2500)
print("giới hạn 1000 :", goi_parse_voi_gioi_han(chain, 1000))
print("giới hạn 50000:", goi_parse_voi_gioi_han(chain, 50_000))
assert goi_parse_voi_gioi_han(chain, 1000) == "RecursionError"
assert goi_parse_voi_gioi_han(chain, 50_000) is True

giới hạn 1000 : RecursionError
giới hạn 50000: True


**Vì sao sai/nguy hiểm:** chuỗi xương cá sâu `~ 2500` vượt giới hạn đệ quy mặc định 1000 của Python. **Code đúng:** dùng cách lặp `O(1)` phụ (đếm chỗ trống), không phụ thuộc độ sâu.

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "**kiểm tra chuỗi tiền thứ tự có hợp lệ, không dựng cây**" → **đếm chỗ trống** (`slots = 1`, mỗi phần tử `-1`, nút thật `+2`).
- Hai kiểm tra bắt buộc: **không âm giữa chừng** và **bằng `0` ở cuối**.
- Muốn trực giác cấu trúc → **ngăn xếp rút gọn `x # #` thành `#`** hoặc **ngăn xếp bộ đếm con**.
- Phân biệt giá trị nhiều chữ số: **tách theo dấu phẩy**.
- Họ hàng: **Serialize and Deserialize Binary Tree** (297), **Construct Binary Tree from Preorder and Inorder** (105), **Validate Binary Search Tree** (98), **Verify Preorder Sequence in BST** (255), bài đếm bậc vào/bậc ra (đồ thị).
- Thử: `"#"`, `"1,#,#"`, `"#,#,1"`, `"1,#"`, `"9,#,#,1"`.

## 15. Biến thể của bài

1. **Chuỗi duyệt hậu thứ tự** (postorder) hợp lệ: đọc ngược để đưa về tiền thứ tự (đảo chuỗi, đảo vai trái/phải).
2. **Cây `k` nhánh** (mỗi nút thật có `k` con): `slots += k` thay vì `+= 2`.
3. **Trả vị trí lỗi đầu tiên** thay vì `True/False`.
4. **Đếm số cây hợp lệ** có `m` nút thật: số Catalan `C(m)`.
5. **Rừng**: chuỗi gồm nhiều cây liên tiếp (đếm bao nhiêu cây).
#### Code demo

In [22]:
def valid_k_ary(preorder, k):
    slots = 1
    for tok in preorder.split(","):
        slots -= 1
        if slots < 0:
            return False
        if tok != "#":
            slots += k
    return slots == 0


def first_error(preorder):
    slots = 1
    for idx, tok in enumerate(preorder.split(",")):
        slots -= 1
        if slots < 0:
            return idx                                      # vị trí phần tử thừa
        if tok != "#":
            slots += 2
    return -1 if slots == 0 else "thiếu"


def valid_postorder(postorder):
    items = postorder.split(",")[::-1]                       # hậu thứ tự đảo = tiền thứ tự (phải trước trái)
    return valid_slots(",".join(items))


print(valid_k_ary("1,#,#,#", 3), valid_k_ary("1,#,#", 3))      # True False
print(first_error("9,#,#,1"), first_error("1,#"), first_error("1,#,#"))   # 3 thiếu -1
print(valid_postorder("#,#,1,#,#,2,3"), valid_postorder("#,#,1"))

True False
3 thiếu -1
True True


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Phân tích đệ quy | văn phạm `S → #` hoặc `x S S` | `O(n)` | `O(h)` | baseline, dễ hiểu |
| 2. Ngăn xếp rút gọn | `x # #` → `#` | `O(n)` | `O(n)` | trực giác cấu trúc |
| 3. Đếm chỗ trống | `slots -1`, nút thật `+2` | `O(n)` | `O(1)` | **bản nộp chuẩn** |
| 4. Ngăn xếp bộ đếm | số con còn thiếu | `O(n)` | `O(h)` | mô phỏng duyệt cây |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán `oracle` (hợp lệ hay không): (a) `"#"`; (b) `"1,#,#"`; (c) `"1,#,#,#"`; (d) `"1,2,#,#,#"`.

**Bài 2 (tính bằng tay).** Với `"9,#,#,1"` và `"1,#"`, ghi `slots` sau mỗi phần tử và kết luận.

**Bài 3 (điền TODO).** Hoàn thiện `valid_todo(preorder)` theo Cách 3.
#### Code demo

In [23]:
def valid_todo(preorder):
    # TODO 1: slots = 1
    # TODO 2: với mỗi tok trong preorder.split(","): slots -= 1; nếu slots < 0 thì trả False
    # TODO 3: nếu tok != "#" thì slots += 2
    # TODO 4: return slots == 0
    pass


# Tự làm xong rồi mới mở test:
# for text, expected in examples: assert valid_todo(text) == expected

**Bài 4 (tự viết lại).** Viết `count_nodes_if_valid(preorder)` trả số **nút thật** nếu chuỗi hợp lệ, ngược lại `-1`.

**Bài 5 (biến thể / mini challenge).** Viết `count_trees(preorder)` cho một chuỗi gồm **nhiều cây liên tiếp** (rừng): trả số cây nếu hợp lệ, ngược lại `-1`. Gợi ý: mỗi khi `slots` về `0`, một cây vừa kết thúc; bắt đầu cây mới với `slots = 1`. Thử `"1,#,#,2,#,#,#"` → `3`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: mỗi nút thật cần hai con. Bài 2: bắt đầu `slots = 1`. Bài 4: đếm khi gặp phần tử khác `#`. Bài 5: một cây xong khi `slots` về `0`.
- **Mức 2 (vừa):** Bài 1: `True`, `True`, `False`, `True`. Bài 2: `"9,#,#,1"`: `1→2→1→0→-1` thừa; `"1,#"`: `1→2→1` kết thúc `1` ≠ 0 thiếu. Bài 3: khung Cách 3.
- **Mức 3 (sâu):** Bài 4: cộng thêm biến `nodes`, kết thúc trả `nodes if slots == 0 else -1`. Bài 5: khi `slots == 0` và còn phần tử thì `trees += 1; slots = 1` **trước** khi trừ phần tử tiếp theo; cuối cùng `slots == 0` thì tính cây cuối.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [24]:
# Bài 1
assert [oracle(t) for t in ("#", "1,#,#", "1,#,#,#", "1,2,#,#,#")] == [True, True, False, True]

# Bài 2
assert trace_slots_quick("9,#,#,1")[-1] == ("1", "hết chỗ")
assert trace_slots_quick("1,#")[-1] == ("#", 1)


# Bài 3
def valid_todo_ans(preorder):
    slots = 1
    for tok in preorder.split(","):
        slots -= 1
        if slots < 0:
            return False
        if tok != "#":
            slots += 2
    return slots == 0


for text, expected in examples:
    assert valid_todo_ans(text) == expected


# Bài 4
def count_nodes_if_valid(preorder):
    slots, nodes = 1, 0
    for tok in preorder.split(","):
        slots -= 1
        if slots < 0:
            return -1
        if tok != "#":
            slots += 2
            nodes += 1
    return nodes if slots == 0 else -1


assert count_nodes_if_valid("9,3,4,#,#,1,#,#,2,#,6,#,#") == 6
assert count_nodes_if_valid("1,#") == -1 and count_nodes_if_valid("#") == 0


# Bài 5
def count_trees(preorder):
    slots, trees = 1, 0
    for tok in preorder.split(","):
        if slots == 0:                        # cây trước đã đủ -> bắt đầu cây mới
            trees += 1
            slots = 1
        slots -= 1
        if tok != "#":
            slots += 2
    if slots != 0:
        return -1
    return trees + 1


assert count_trees("1,#,#,2,#,#,#") == 3
assert count_trees("#") == 1 and count_trees("1,#") == -1 and count_trees("1,#,#,#,#") == 3
print("PASS đáp án bài tập")

PASS đáp án bài tập


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đếm chỗ trống)

```python
class Solution:
    def isValidSerialization(self, preorder: str) -> bool:
        slots = 1
        for tok in preorder.split(","):
            slots -= 1
            if slots < 0:
                return False
            if tok != "#":
                slots += 2
        return slots == 0
```

**Checklist**

- [ ] Khởi tạo `slots = 1` (chỗ cho nút gốc)?
- [ ] Mỗi phần tử `slots -= 1`, nút thật `slots += 2`, `#` không cộng?
- [ ] Kiểm tra **âm giữa chừng** (`slots < 0 → False`)?
- [ ] Kiểm tra **bằng 0 ở cuối**?
- [ ] Tách theo **dấu phẩy** (giá trị nhiều chữ số)?
- [ ] Không dựng lại cây (đúng yêu cầu của đề)?
- [ ] Đã thử `"#"`, `"1,#,#"`, `"#,#,1"`, `"1,#"`, `"9,#,#,1"` và chuỗi dài 10^4 ký tự?
- [ ] Skeleton đúng `class Solution` / `def isValidSerialization(self, preorder: str) -> bool`?